# Markov Decision Process (MDP)

### Lava environments
The environments used are LavaFloor (visible in the figure) and its variations.

![Lava](images/lava.png)

The agent starts in cell $(0, 0)$ and has to reach the treasure in $(2, 3)$. In addition to the walls of the previous environments, the floor is covered with lava, there is a black pit of death.

Moreover, the agent can't comfortably perform its actions that instead have a stochastic outcome (visible in the figure):

![Dynact](images/dynact.png)

The action dynamics is the following:
- $P(0.8)$ of moving **in the desired direction**
- $P(0.1)$ of moving in a direction 90° with respect to the desired direction

Finally, since the floor is covered in lava, the agent receives a negative reward for each of its steps!

- -0.04 for each lava cell (L)
- -5 for the black pit (P). End of episode
- +1 for the treasure (G). End of episode

In [1]:
import os, sys, random
module_path = os.path.abspath(os.path.join('../tools'))
if module_path not in sys.path:
    sys.path.append(module_path)

import gym, envs
from utils.ai_lab_functions import *
from timeit import default_timer as timer
from tqdm import tqdm as tqdm

### Value Iteration

In [2]:
def value_iteration(environment, maxiters=300, discount=0.9, max_error=1e-3):
    U_1 = [0 for _ in range(environment.observation_space.n)] # vector of utilities for states S
    iter_count = 0
    
    while True:
        iter_count += 1
        
        delta = 0 # maximum change in the utility of any state in an iteration
        U = U_1.copy()
        
        for state in range(0, environment.observation_space.n):
            if environment.grid[state] == "P" or environment.grid[state] == "G":
                U_1[state] = environment.RS[state]
            else:
                action_utilities = [U[state] for _ in range(0, environment.action_space.n)]
                for action in range(0, environment.action_space.n):
                    u_sum = 0
                    for next_state in range(0, environment.observation_space.n):
                        u_sum += environment.T[state, action, next_state] * U[next_state]
                    action_utilities.append(u_sum)
                
                U_1[state] = environment.RS[state] + discount * max(action_utilities)
                
            delta = max(delta, abs(U_1[state] - U[state]))
            
        if delta < max_error * (1 - discount) / discount or iter_count >= maxiters:
            break

    return values_to_policy(np.asarray(U), environment) # automatically convert the value matrix U to a policy

In [3]:
env_name = "LavaFloor-v0"
#env_name = "HugeLavaFloor-v0"
#env_name = "NiceLavaFloor-v0"
#env_name = "VeryBadLavaFloor-v0"



env = gym.make(env_name)
print("\nENV RENDER:")
env.render()

print("Transition model for ", env_name, " : ") #assume transition functions is the same for all states
state=0
next_state=1
for i in range(0,env.action_space.n):
    print("probability of reaching ", env.state_to_pos(next_state), "from ", env.state_to_pos(state), " executing action ", env.actions[i], " : ", env.T[state, i, next_state])
print("Reward for non terminal states: ",env.RS[env.pos_to_state(0,0)]) #assume all states have same reward
for state in range(0,env.observation_space.n):
    if env.grid[state] == "P" or env.grid[state] == "G":
                    print("Reward for state :", env.state_to_pos(state) ," (state type: ", env.grid[state],") : ",env.RS[state])

t = timer()
policy = value_iteration(env)

print("\nEXECUTION TIME: \n{}".format(round(timer() - t, 4)))
policy_render = np.vectorize(env.actions.get)(policy.reshape(env.rows, env.cols))
results = CheckResult_L3(env_name, policy_render)
results.check_value_iteration()


ENV RENDER:
[['S' 'L' 'L' 'L']
 ['L' 'W' 'L' 'P']
 ['L' 'L' 'L' 'G']]
Transition model for  LavaFloor-v0  : 
probability of reaching  (0, 1) from  (0, 0)  executing action  L  :  0.0
probability of reaching  (0, 1) from  (0, 0)  executing action  R  :  0.8
probability of reaching  (0, 1) from  (0, 0)  executing action  U  :  0.1
probability of reaching  (0, 1) from  (0, 0)  executing action  D  :  0.1
Reward for non terminal states:  -0.04
Reward for state : (1, 3)  (state type:  P ) :  -5.0
Reward for state : (2, 3)  (state type:  G ) :  1.0

EXECUTION TIME: 
0.0074

#################################################################
#######  Environment: LavaFloor-v0 	Value Iteration  ########
#################################################################

===> Your solution is correct!

Policy:
[['D' 'L' 'L' 'U']
 ['D' 'L' 'L' 'L']
 ['R' 'R' 'R' 'L']]
